In [114]:
!pip install matplotlib scipy pandas cvxpy tqdm seaborn kaggle 'cvxpy[glpk]' polarix axelrod -q

In [115]:
import axelrod as axl                                                                                                                                                     
import numpy as np
import polarix as plx
import jax.numpy as jnp
import sys        
import pickle                                                                                                                                                         
sys.path.insert(0, "/Users/gabesmithline/Desktop/Causal-Game-Analysis")
from src.iterative_game_analysis.metagame import MetaGame
import os
import pandas as pd


In [ ]:
players = [                                 
    # Cooperative                       
    axl.TitForTat(),
    axl.TitFor2Tats(),                                                                                                                                                                            
    axl.Cooperator(),
    axl.GeneralSoftGrudger(),                                                                                                                                                                     
    axl.WinStayLoseShift(),
    axl.Grudger(),
    # Exploitative
    axl.Defector(),
    axl.SuspiciousTitForTat(),
    axl.HardGoByMajority(),
    axl.Bully(),
    axl.Aggravater(),
    axl.Predator(),
    axl.BackStabber(),
    axl.DoubleCrosser(),
    # Mixed/Adaptive
    axl.Random(),
    axl.TwoTitsForTat(),
    axl.HardTitForTat(),
    axl.Prober(),
    axl.SoftJoss(),
    axl.Prober2(),
    axl.Prober3(),
    # Sophisticated/ZD
    axl.Calculator(),
    axl.Punisher(),
    axl.InversePunisher(),
    axl.AdaptiveTitForTat(),
    axl.EvolvedFSM16(),
    axl.EvolvedFSM4(),
    axl.ThueMorse(),
    axl.Detective(),
    axl.TrickyDefector(),
]

strategy_names = [str(p) for p in players]
print(f"Strategies: {len(players)}")


Strategies: 30


In [117]:
print([s.name for s in axl.filtered_strategies({'cooperates_against_cooperator': False})])

['ALLCorALLD', 'AON2', 'Adaptive Pavlov 2006', 'Adaptive Pavlov 2011', 'Adaptive', 'Adaptive Tit For Tat', 'AdaptorBrief', 'AdaptorLong', 'Aggravater', 'Alexei', 'Alternator', 'Alternator Hunter', 'AntiCycler', 'Anti Tit For Tat', 'Appeaser', 'Arrogant QLearner', 'Average Copier', 'BackStabber', 'Better and Better', 'Bully', 'Burn Both Ends', 'Bush Mosteller', 'Calculator', 'CAPRI', 'Cautious QLearner', 'CollectiveStrategy', 'Contrite Tit For Tat', 'Cooperator', 'Cooperator Hunter', 'Cycle Hunter', 'Cycler CCCCCD', 'Cycler CCCD', 'Cycler CCCDCD', 'Cycler CCD', 'Cycler DC', 'Cycler DDC', 'DBS', 'Darwin', 'Defector', 'Defector Hunter', 'Delayed AON1', 'Desperate', 'Detective', 'DoubleCrosser', 'DoubleResurrection', 'Doubler', 'Dynamic Two Tits For Tat', 'EasyGo', 'EugineNier', 'Eventual Cycle Hunter', 'Evolved ANN', 'Evolved ANN 5', 'Evolved ANN 5 Noise 05', 'Evolved FSM 16', 'Evolved FSM 16 Noise 05', 'Evolved FSM 4', 'Evolved FSM 6', 'Evolved HMM 5', 'EvolvedLookerUp1_1_1', 'EvolvedLoo

In [118]:
strats = axl.filtered_strategies({
    'long_run_time': False,
    'max_memory_depth': 10,
})
#players = [s() for s in strats[:20]]
strategy_names = [str(p) for p in players]
n = len(players)
print(f"Strategies: {n}")
turns = 200 #length of each matchup
reptitions = 200 # number of matchups each pair plays independently 
noise = .01 # % change each action gets flipped (intended C becomes D, or vice versa). This breaks deterministic strategies and differentiates robust vs fragile cooperators

'''
axl.Game(r, s, t, p) lets you set any 2x2 symmetric game:

           C        D
  C     (r,r)    (s,t)
  D     (t,s)    (p,p)

  So you can do:

  - PD: t > r > p > s (e.g., t=5, r=3, p=1, s=0)
  - Hawk-Dove/Chicken: t > r > s > p (e.g., t=5, r=3, s=1, p=0)
  - Stag Hunt: r > t > p > s (e.g., r=5, t=3, p=1, s=0)
  - Coordination: r > t, p > s (e.g., r=5, t=0, p=3, s=0)
'''
#(C, C) cell -> payoff when you both cooperate
r = 7
#(D, D) cell -> payoff when you both defect 
p = 4
#(D, C) cell -> payoff when you defect and opponent cooperates
t = 10
#(C, D) cell -> payoff when you cooperate and opponent defects 
s=0
game = axl.Game(r=r, s=s, t=t, p=p)  # higher temptation

tournament = axl.Tournament(players, turns=turns, repetitions=reptitions, noise=noise, game=game)
results = tournament.play()

# Per-repetition payoff and cooperation data for bootstrapping
# interactions[(i,j)] = list of repetitions, each a list of (C/D, C/D) turns
n_reps = len(results.payoffs[0][0])
payoff_reps = np.zeros((n_reps, n, n))
coop_reps = np.zeros((n_reps, n, n))


for i in range(n):
    for j in range(n):
        payoff_reps[:, i, j] = results.payoffs[i][j]

# For cooperation, normalised_cooperation is only the mean
# Check if per-rep cooperation is available
print(f"cooperation shape: {np.array(results.cooperation).shape}")
print(f"cooperation[0][0]: {results.cooperation[0][0]}")

# Mean matrices
payoff_matrix = payoff_reps.mean(axis=0)
coop_matrix = np.array(results.normalised_cooperation)

# NW matrix: geometric mean of both players' scores
nw_matrix = np.sqrt(payoff_matrix * payoff_matrix.T)

print(f"Payoff matrix: [{payoff_matrix.min():.2f}, {payoff_matrix.max():.2f}]")
print(f"Cooperation matrix: [{coop_matrix.min():.2f}, {coop_matrix.max():.2f}]")
print(f"NW matrix: [{nw_matrix.min():.2f}, {nw_matrix.max():.2f}]")

# Save everything for bootstrapping
data = {
    'strategy_names': strategy_names,
    'payoff_reps': payoff_reps,      # (100, n, n) - bootstrap over axis 0
    'coop_reps': coop_reps,          # (100, n, n)
    'payoff_matrix': payoff_matrix,  # (n, n) mean
    'coop_matrix': coop_matrix,      # (n, n) mean
    'nw_matrix': nw_matrix,          # (n, n) mean
    'n_strategies': n,
    'turns': turns,
    'repetitions': reptitions,
    'noise': noise
}

os.makedirs('pd_data', exist_ok=True)
with open('pd_data/pd_tournament.pkl', 'wb') as f:
    pickle.dump(data, f)

print(f"Saved to data/pd_tournament.pkl")



Strategies: 30


Playing matches:   6%|▌         | 29/465 [00:10<02:29,  2.91it/s]

ValueError: 

In [ ]:
print(f"Strategies: {len(strategy_names)}")                                                                                                                               
print(f"Matrix shape: {payoff_matrix.shape}")

Strategies: 20
Matrix shape: (20, 20)


In [ ]:
mg_full = MetaGame(policies=strategy_names, payoff_matrix=payoff_matrix)
sigma_full = mg_full.solve("mene")

support = [(name, sigma_full[i]) for i, name in enumerate(strategy_names) if sigma_full[i] > 1e-3]
print(f"Equilibrium support: {len(support)} strategies")
for name, w in sorted(support, key=lambda x: -x[1]):
    print(f"  {name}: {w:.4f}")



Equilibrium support: 3 strategies
  Tit For 2 Tats: 0.6257
  Soft Joss: 0.9: 0.2117
  Evolved FSM 16: 0.1625


In [ ]:
df = pd.DataFrame(payoff_matrix, index=strategy_names, columns=strategy_names)
print(df.round(2).to_string())

                                   Tit For Tat  Tit For 2 Tats  Cooperator  General Soft Grudger: n=1,d=4,c=2  Win-Stay Lose-Shift  Grudger  Defector  Suspicious Tit For Tat  Hard Go By Majority  Bully  Random: 0.5  Two Tits For Tat  Hard Tit For Tat  Prober  Soft Joss: 0.9  Calculator  Punisher  Inverse Punisher  Adaptive Tit For Tat: 0.5  Evolved FSM 16
Tit For Tat                               5.77            6.99        6.99                               6.76                 5.64     4.77      3.96                    5.13                 5.32   5.23         5.24              4.70              4.77    5.62            6.50        5.05      4.83              4.69                       5.84            5.95
Tit For 2 Tats                            6.89            6.96        6.97                               6.79                 5.01     5.11      3.90                    6.85                 6.89   4.17         4.37              5.52              5.09    4.04            6.90        4.

In [ ]:
df_coop = pd.DataFrame(coop_matrix, index=strategy_names, columns=strategy_names)
print("\nCooperation Matrix:")
print(df_coop.round(2).to_string())    


Cooperation Matrix:
                                   Tit For Tat  Tit For 2 Tats  Cooperator  General Soft Grudger: n=1,d=4,c=2  Win-Stay Lose-Shift  Grudger  Defector  Suspicious Tit For Tat  Hard Go By Majority  Bully  Random: 0.5  Two Tits For Tat  Hard Tit For Tat  Prober  Soft Joss: 0.9  Calculator  Punisher  Inverse Punisher  Adaptive Tit For Tat: 0.5  Evolved FSM 16
Tit For Tat                               0.66            0.98        0.98                               0.91                 0.61     0.28      0.02                    0.47                 0.49   0.50         0.50              0.26              0.28    0.61            0.86        0.44      0.31              0.26                       0.67            0.72
Tit For 2 Tats                            0.99            0.99        0.99                               0.96                 0.71     0.42      0.04                    0.99                 0.98   0.60         0.75              0.77              0.41    0.09     

In [ ]:
#holdout analysis:
with open('pd_data/pd_tournament.pkl', 'rb') as f:
    pd_data = pickle.load(f)
strategy_names = pd_data['strategy_names']
payoff_reps = pd_data['payoff_reps']       # (100, n, n)
coop_matrix = pd_data['coop_matrix']        # (n, n) mean cooperation
n = len(strategy_names)
n_bootstrap = 100
rng = np.random.default_rng(42)

metrics = ['payoff', 'coop']
metric_labels = {'payoff': 'delta Payoff', 'nw': 'delta NW', 'coop': 'delta Coop'}

# Storage: full and LOO welfare per bootstrap
results = {m: {'full': []} for m in metrics}
for s in strategy_names:
    for m in metrics:
        results[m][f'loo_{s}'] = []

# Store sigmas for support filtering
sigma_samples = []

n_reps = payoff_reps.shape[0]
for b in range(n_bootstrap):
    
    boot_payoff = np.zeros((n, n)) #resample repititions indepdently
    for i in range(n):
        for j in range(n):
            idx = rng.choice(n_reps, size=n_reps, replace=True)
            boot_payoff[i, j] = payoff_reps[idx, i, j].mean()
    boot_payoff_sym = (boot_payoff + boot_payoff.T) / 2

    #Full game equilibrium for bootstrap
    mg = MetaGame(policies=strategy_names, payoff_matrix=boot_payoff_sym)
    sigma = mg.solve("mene")
    sigma_samples.append(sigma)

    # Evaluate metrics at equilibrium
    results['payoff']['full'].append(float(sigma @ boot_payoff_sym @ sigma))
    #results['nw']['full'].append(float(sigma @ nw_matrix @ sigma))
    results['coop']['full'].append(float(sigma @ coop_matrix @ sigma))

    # LOO
    for i, s in enumerate(strategy_names):
        loo_idx = [j for j in range(n) if j != i]
        loo_payoff = boot_payoff_sym[np.ix_(loo_idx, loo_idx)]
        loo_names = [strategy_names[j] for j in loo_idx]

        mg_loo = MetaGame(policies=loo_names, payoff_matrix=loo_payoff)
        sigma_loo = mg_loo.solve("mene")

        results['payoff'][f'loo_{s}'].append(float(sigma_loo @ loo_payoff @ sigma_loo))
        #results['nw'][f'loo_{s}'].append(float(sigma_loo @ nw_matrix[np.ix_(loo_idx, loo_idx)] @ sigma_loo))
        results['coop'][f'loo_{s}'].append(float(sigma_loo @ coop_matrix[np.ix_(loo_idx, loo_idx)] @ sigma_loo))

    if (b + 1) % 100 == 0:
        print(f"  {b+1}/{n_bootstrap} done")

sigmas = np.array(sigma_samples)

# Report
header = '| **Strategy** | **n** | ' + ' | '.join(f'**{metric_labels[m]}**' for m in metrics) + ' |'
sep = '| --- | --- | ' + ' | '.join('---' for _ in metrics) + ' |'
print(header)
print(sep)

for s_idx, s in enumerate(strategy_names):
    mask = sigmas[:, s_idx] >= 10e-13
    n_active = int(mask.sum())
    if n_active < 5:
        continue

    row = f'| {s} | {n_active} |'
    for m in metrics:
        full_vals = np.array(results[m]['full'])[mask]
        loo_vals = np.array(results[m][f'loo_{s}'])[mask]
        diffs = np.round(full_vals - loo_vals, 6)

        mean = np.mean(diffs)
        lo, hi = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

        stars = ''
        if lo > 0 or hi < 0:
            stars = '**'

        row += f' {mean:+.4f} [{lo:.4f}, {hi:.4f}]{stars} |'
    print(row)

/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000019 > 1e-05).
  warnings.warn(


  100/100 done
| **Strategy** | **n** | **delta Payoff** | **delta Coop** |
| --- | --- | --- | --- |
| Cooperator | 100 | +0.0713 [0.0158, 0.3175]** | +0.0266 [0.0144, 0.0834]** |
| Win-Stay Lose-Shift | 97 | -0.3851 [-0.4643, -0.3526]** | -0.1015 [-0.1205, -0.1084]** |
| Soft Joss: 0.9 | 74 | -0.0683 [-0.4499, 0.0357] | -0.0231 [-0.1147, -0.0003]** |


In [ ]:
#interaction effects
from itertools import combinations

n_bootstrap = 100
n_reps = payoff_reps.shape[0]
rng = np.random.default_rng(42)

#test_strategies = strategy_names  # or pick a subset
test_strategies = ['Cooperator', 'Win-Stay Lose-Shift', 'Soft Joss: 0.9',
                     'Tit For 2 Tats', 'Evolved FSM 16']

interaction_results = {}

for b in range(n_bootstrap):
    boot_payoff = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            idx = rng.choice(n_reps, size=n_reps, replace=True)
            boot_payoff[i, j] = payoff_reps[idx, i, j].mean()
    boot_payoff_sym = (boot_payoff + boot_payoff.T) / 2

    # Full game
    mg = MetaGame(policies=strategy_names, payoff_matrix=boot_payoff_sym)
    sigma = mg.solve("mene")
    W_full = float(sigma @ boot_payoff_sym @ sigma)
    C_full = float(sigma @ coop_matrix @ sigma)

    # Cache LOO results
    loo_cache = {}
    for i, s in enumerate(strategy_names):
        loo_idx = [j for j in range(n) if j != i]
        loo_payoff = boot_payoff_sym[np.ix_(loo_idx, loo_idx)]
        loo_names = [strategy_names[j] for j in loo_idx]
        mg_loo = MetaGame(policies=loo_names, payoff_matrix=loo_payoff)
        sigma_loo = mg_loo.solve("mene")
        loo_cache[s] = {
            'payoff': float(sigma_loo @ loo_payoff @ sigma_loo),
            'coop': float(sigma_loo @ coop_matrix[np.ix_(loo_idx, loo_idx)] @ sigma_loo),
        }

    # LTO pairs
    for s_a, s_b in combinations(strategy_names, 2):
        i_a = strategy_names.index(s_a)
        i_b = strategy_names.index(s_b)
        lto_idx = [j for j in range(n) if j != i_a and j != i_b]
        lto_payoff = boot_payoff_sym[np.ix_(lto_idx, lto_idx)]
        lto_names = [strategy_names[j] for j in lto_idx]
        mg_lto = MetaGame(policies=lto_names, payoff_matrix=lto_payoff)
        sigma_lto = mg_lto.solve("mene")
        W_lto = float(sigma_lto @ lto_payoff @ sigma_lto)
        C_lto = float(sigma_lto @ coop_matrix[np.ix_(lto_idx, lto_idx)] @ sigma_lto)

        key = (s_a, s_b)
        if key not in interaction_results:
            interaction_results[key] = {'payoff': [], 'coop': []}

        # Harsanyi dividend
        for m, W_f, W_a, W_b, W_ab in [
            ('payoff', W_full, loo_cache[s_a]['payoff'], loo_cache[s_b]['payoff'], W_lto),
            ('coop', C_full, loo_cache[s_a]['coop'], loo_cache[s_b]['coop'], C_lto),
        ]:
            dividend = W_f - W_a - W_b + W_ab
            interaction_results[key][m].append(dividend)

    if (b + 1) % 10 == 0:
        print(f"  {b+1}/{n_bootstrap} done")

# Report
print(f"\n{'Pair':<55} {'Δ² Payoff':>12} {'95% CI':>24} {'Δ² Coop':>12} {'95% CI':>24}")
print("-" * 130)

for (s_a, s_b), res in sorted(interaction_results.items(), key=lambda x: -abs(np.mean(x[1]['payoff']))):
    for m in ['payoff', 'coop']:
        vals = np.array(res[m])

    p_vals = np.array(res['payoff'])
    c_vals = np.array(res['coop'])

    p_mean = np.mean(p_vals)
    p_lo, p_hi = np.percentile(p_vals, 2.5), np.percentile(p_vals, 97.5)
    p_sig = '**' if p_lo > 0 or p_hi < 0 else ''

    c_mean = np.mean(c_vals)
    c_lo, c_hi = np.percentile(c_vals, 2.5), np.percentile(c_vals, 97.5)
    c_sig = '**' if c_lo > 0 or c_hi < 0 else ''

    # if abs(p_mean) > 0.001 or abs(c_mean) > 0.001:  # filter tiny effects
    print(f"  {s_a + ' × ' + s_b:<55} {p_mean:>+.4f} [{p_lo:>+.4f}, {p_hi:>+.4f}]{p_sig:>3}   {c_mean:>+.4f} [{c_lo:>+.4f}, {c_hi:>+.4f}]{c_sig:>3}")

  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done

Pair                                                       Δ² Payoff                   95% CI      Δ² Coop                   95% CI
----------------------------------------------------------------------------------------------------------------------------------
  Tit For 2 Tats × Win-Stay Lose-Shift                    -1.4908 [-1.7015, -0.0011] **   -0.4342 [-0.4917, -0.0007] **
  Cooperator × Win-Stay Lose-Shift                        -1.3813 [-1.6714, +0.0635]      -0.3961 [-0.4770, +0.0248]   
  Cooperator × Soft Joss: 0.9                             -0.1665 [-0.5579, -0.0759] **   -0.0557 [-0.1529, -0.0384] **
  Tit For 2 Tats × Soft Joss: 0.9                         -0.0565 [-0.4535, +0.0000]      -0.0141 [-0.1086, +0.0000]   
  Win-Stay Lose-Shift × Soft Joss: 0.9                    -0.0506 [-0.4415, +0.0322]      -0.0171 [-0.1145, +0.0000